# Confidence Analysis — Fine-tuned vs. Vanilla Models

Compares `student_confidence` across four model variants:
- **LLaMA (Vanilla)** — `vanilla_llama.json`
- **Mistral (Vanilla)** — `vanilla_ministral.json`
- **LLaMA (Fine-tuned / QLoRA)** — `qlora_llama.json`
- **Mistral (Fine-tuned / QLoRA)** — `qlora_mistral.json`

Analysis sections:
1. Overall confidence distribution
2. Confidence vs. correctness
3. Confidence per Level 1 category (service type)
4. Confidence per Level 2 category (product)
5. Confidence per Level 3 category (scenario)
6. Calibration — does confidence predict accuracy?
7. Fallback & format compliance interactions

In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
FIGSIZE = (12, 5)
COLORS = sns.color_palette('Set2', 4)

DATA_PATH = Path('../../data/results')

In [ ]:
# ── Load all four result files ──────────────────────────────────────────────
FILES = {
    'LLaMA Vanilla':   'vanilla_llama.json',
    'Mistral Vanilla': 'vanilla_ministral.json',
    'LLaMA Fine-tuned':   'qlora_llama.json',
    'Mistral Fine-tuned': 'qlora_mistral.json',
}

dfs = {}
for model_name, fname in FILES.items():
    with open(DATA_PATH / fname) as f:
        raw = json.load(f)
    dfs[model_name] = pd.DataFrame(raw['data'], columns=raw['columns'])
    print(f'{model_name:25s}  {len(dfs[model_name]):>5} rows')

MODEL_ORDER = list(FILES.keys())

In [ ]:
# ── Parse hierarchical label into Level 1 / 2 / 3 ──────────────────────────
# Label format: "(1X Service Type, 2X Product), 3 Scenario"
_LABEL_RE = re.compile(r'\(([^,]+),\s*([^)]+)\),\s*(.+)')

def parse_label(label):
    if not isinstance(label, str):
        return None, None, None
    m = _LABEL_RE.match(label)
    if m:
        return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
    return None, None, None

for df in dfs.values():
    df[['level1', 'level2', 'level3']] = (
        df['true_label']
        .apply(lambda x: pd.Series(parse_label(x), index=['level1','level2','level3']))
    )
    df['correct'] = df['correct'].astype(bool)
    df['is_fallback'] = df['is_fallback'].astype(bool)
    df['format_compliant'] = df['format_compliant'].astype(bool)

# Combined long-format DataFrame for faceted plots
combined = pd.concat(
    [df.assign(model=name) for name, df in dfs.items()],
    ignore_index=True
)
combined['model'] = pd.Categorical(combined['model'], categories=MODEL_ORDER, ordered=True)
combined.head(2)

---
## 1 · Overall Confidence Distribution

In [ ]:
# Summary statistics
stats = (
    combined.groupby('model')['student_confidence']
    .agg(['mean','median','std',
          lambda s: s.quantile(0.25),
          lambda s: s.quantile(0.75),
          'min','max'])
    .rename(columns={
        'mean':'Mean','median':'Median','std':'Std',
        '<lambda_0>':'Q25','<lambda_1>':'Q75','min':'Min','max':'Max'
    })
)
stats.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE
ax = axes[0]
for (name, df), color in zip(dfs.items(), COLORS):
    df['student_confidence'].plot.kde(ax=ax, label=name, color=color, lw=2)
ax.set_xlabel('Confidence')
ax.set_title('Confidence Distribution (KDE)')
ax.legend()

# Box plot
ax = axes[1]
combined.boxplot(column='student_confidence', by='model', ax=ax,
                 patch_artist=True,
                 boxprops=dict(facecolor='lightsteelblue'),
                 medianprops=dict(color='navy', lw=2))
ax.set_title('Confidence Box Plot')
ax.set_xlabel('')
ax.set_ylabel('Confidence')
plt.suptitle('')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## 2 · Confidence vs. Correctness

In [ ]:
# Mean confidence split by correct / incorrect
conf_by_correct = (
    combined.groupby(['model','correct'])['student_confidence']
    .mean()
    .unstack('correct')
    .rename(columns={True:'Correct', False:'Incorrect'})
)
conf_by_correct.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar: mean confidence correct vs incorrect
conf_by_correct.plot.bar(ax=axes[0], color=['steelblue','tomato'], edgecolor='white')
axes[0].set_title('Mean Confidence: Correct vs Incorrect')
axes[0].set_xlabel('')
axes[0].set_ylabel('Mean Confidence')
axes[0].tick_params(axis='x', rotation=15)
axes[0].set_ylim(0, 1.05)
axes[0].legend(title='Prediction')

# Box plot faceted by correctness
correct_map = {True: 'Correct', False: 'Incorrect'}
combined['Prediction'] = combined['correct'].map(correct_map)
sns.boxplot(
    data=combined, x='model', y='student_confidence',
    hue='Prediction', palette={'Correct':'steelblue','Incorrect':'tomato'},
    ax=axes[1]
)
axes[1].set_title('Confidence Distribution by Correctness')
axes[1].set_xlabel('')
axes[1].set_ylabel('Confidence')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

---
## 3 · Confidence per Level 1 Category (Service Type)

In [ ]:
def level_summary(level_col, min_count=10):
    """Mean confidence and accuracy per label category, per model."""
    grp = combined.groupby(['model', level_col])
    mean_conf = grp['student_confidence'].mean().rename('mean_confidence')
    accuracy  = grp['correct'].mean().rename('accuracy')
    counts    = grp.size().rename('n')
    summary = pd.concat([mean_conf, accuracy, counts], axis=1).reset_index()
    # Keep categories that appear at least min_count times across all models
    valid_cats = (
        summary.groupby(level_col)['n'].sum()
        .loc[lambda s: s >= min_count * len(FILES)]
        .index
    )
    return summary[summary[level_col].isin(valid_cats)]

l1_summary = level_summary('level1')
l1_summary.head(8)

In [ ]:
# Pivot for heatmap
def conf_heatmap(summary_df, level_col, title):
    pivot = summary_df.pivot(index=level_col, columns='model', values='mean_confidence')
    pivot = pivot[MODEL_ORDER]  # consistent column order
    fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.5 + 1)))
    sns.heatmap(
        pivot, annot=True, fmt='.3f', cmap='YlGnBu',
        vmin=0, vmax=1, linewidths=0.4, ax=ax,
        cbar_kws={'label': 'Mean Confidence'}
    )
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
    return pivot

_ = conf_heatmap(l1_summary, 'level1', 'Mean Confidence per Level 1 Category')

In [ ]:
# Grouped bar: confidence per L1 category
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, metric, ylabel, title in [
    (axes[0], 'mean_confidence', 'Mean Confidence', 'Mean Confidence per Level 1'),
    (axes[1], 'accuracy',        'Accuracy',         'Accuracy per Level 1'),
]:
    pivot = l1_summary.pivot(index='level1', columns='model', values=metric)[MODEL_ORDER]
    pivot.plot.bar(ax=ax, edgecolor='white', width=0.75)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=25)
    ax.legend(title='Model', fontsize=8)

plt.tight_layout()
plt.show()

---
## 4 · Confidence per Level 2 Category (Product)

In [ ]:
l2_summary = level_summary('level2', min_count=5)
print(f'Level 2 categories shown: {l2_summary["level2"].nunique()}')
_ = conf_heatmap(l2_summary, 'level2', 'Mean Confidence per Level 2 Category')

In [ ]:
# Scatter: confidence vs accuracy per L2 category (bubble size = count)
fig, axes = plt.subplots(1, len(FILES), figsize=(16, 4), sharey=True, sharex=True)

for ax, (model_name, color) in zip(axes, zip(MODEL_ORDER, COLORS)):
    sub = l2_summary[l2_summary['model'] == model_name]
    sc = ax.scatter(
        sub['accuracy'], sub['mean_confidence'],
        s=sub['n'] * 0.5, c=[color], alpha=0.7, edgecolors='grey', lw=0.5
    )
    # Annotate top categories by count
    for _, row in sub.nlargest(5, 'n').iterrows():
        ax.annotate(
            row['level2'].replace('2g ', '').replace('2a ', '').replace('2 ', ''),
            (row['accuracy'], row['mean_confidence']),
            fontsize=7, ha='center', va='bottom'
        )
    ax.set_title(model_name, fontsize=9)
    ax.set_xlabel('Accuracy')
    ax.set_xlim(0, 1.05)
    ax.set_ylim(0, 1.05)
    ax.plot([0,1],[0,1], 'k--', lw=0.8, alpha=0.4)  # perfect calibration line

axes[0].set_ylabel('Mean Confidence')
fig.suptitle('Confidence vs. Accuracy per Level 2 Category\n(bubble size ∝ # tickets)', y=1.02)
plt.tight_layout()
plt.show()

---
## 5 · Confidence per Level 3 Category (Scenario)

In [ ]:
l3_summary = level_summary('level3', min_count=5)
print(f'Level 3 categories shown: {l3_summary["level3"].nunique()}')

# Top 20 scenarios by sample count for readability
top20_l3 = (
    l3_summary.groupby('level3')['n'].sum()
    .nlargest(20).index
)
l3_top = l3_summary[l3_summary['level3'].isin(top20_l3)]
_ = conf_heatmap(l3_top, 'level3', 'Mean Confidence — Top 20 Level 3 Scenarios')

In [ ]:
# Strip plot: confidence spread within each top-10 scenario, for fine-tuned models
top10_l3 = (
    combined.groupby('level3')['student_confidence'].count()
    .nlargest(10).index
)
sub = combined[
    combined['level3'].isin(top10_l3) &
    combined['model'].isin(['LLaMA Fine-tuned','Mistral Fine-tuned'])
]

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxplot(
    data=sub, x='level3', y='student_confidence',
    hue='model', palette='Set2', ax=ax, width=0.5
)
ax.set_title('Confidence Distribution — Top 10 Scenarios (Fine-tuned Models)')
ax.set_xlabel('')
ax.set_ylabel('Confidence')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Model')
plt.tight_layout()
plt.show()

---
## 6 · Calibration — Does Confidence Predict Accuracy?

A well-calibrated model at confidence 0.8 should be correct ~80 % of the time.
The reliability diagram below shows observed accuracy vs. mean confidence per decile bucket.

In [ ]:
def reliability_data(df, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    df = df.copy()
    df['bin'] = pd.cut(df['student_confidence'], bins=bins, include_lowest=True)
    grp = df.groupby('bin', observed=False)
    return pd.DataFrame({
        'mean_conf': grp['student_confidence'].mean(),
        'accuracy':  grp['correct'].mean(),
        'n':         grp.size(),
    }).dropna()

fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)

for ax, (model_name, df), color in zip(axes.flat, dfs.items(), COLORS):
    rd = reliability_data(df)
    ax.plot([0,1],[0,1], 'k--', lw=1, label='Perfect calibration')
    ax.bar(
        rd['mean_conf'], rd['accuracy'],
        width=0.08, alpha=0.6, color=color, edgecolor='grey', label='Observed accuracy'
    )
    ax.plot(rd['mean_conf'], rd['accuracy'], 'o-', color=color, lw=1.5, ms=5)
    ax.set_title(model_name)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel('Mean Confidence')
    ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8)

fig.suptitle('Reliability Diagrams (10 bins)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Expected Calibration Error (ECE) — lower is better
def ece(df, n_bins=10):
    rd = reliability_data(df, n_bins)
    weights = rd['n'] / rd['n'].sum()
    return (weights * (rd['accuracy'] - rd['mean_conf']).abs()).sum()

ece_scores = {name: ece(df) for name, df in dfs.items()}
ece_df = pd.Series(ece_scores, name='ECE').to_frame()
ece_df['ECE'] = ece_df['ECE'].round(4)
ece_df

---
## 7 · Fallback & Format Compliance

In [ ]:
# Counts and confidence stats per (is_fallback, format_compliant) group
flag_stats = (
    combined
    .groupby(['model','is_fallback','format_compliant'])
    .agg(
        n=('student_confidence','count'),
        mean_confidence=('student_confidence','mean'),
        accuracy=('correct','mean')
    )
    .round(4)
)
flag_stats

In [ ]:
# Confidence distribution split by fallback
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE: fallback vs non-fallback, per model
ax = axes[0]
for (model_name, df), color in zip(dfs.items(), COLORS):
    df[df['is_fallback']]['student_confidence'].plot.kde(
        ax=ax, label=f'{model_name} (fallback)', color=color, lw=2, linestyle='--'
    )
    df[~df['is_fallback']]['student_confidence'].plot.kde(
        ax=ax, label=f'{model_name} (normal)', color=color, lw=2, linestyle='-'
    )
ax.set_title('Confidence: Fallback vs Normal')
ax.set_xlabel('Confidence')
ax.legend(fontsize=7, ncol=2)

# Fallback rate per model
ax = axes[1]
fallback_rates = {
    name: df['is_fallback'].mean() for name, df in dfs.items()
}
ax.bar(fallback_rates.keys(), fallback_rates.values(),
       color=COLORS, edgecolor='white')
ax.set_title('Fallback Rate per Model')
ax.set_ylabel('Fraction of Predictions')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=15)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.show()

---
## Summary Table

In [ ]:
summary_table = pd.DataFrame({
    model: {
        'N': len(df),
        'Accuracy': round(df['correct'].mean(), 4),
        'Mean Confidence': round(df['student_confidence'].mean(), 4),
        'Median Confidence': round(df['student_confidence'].median(), 4),
        'Std Confidence': round(df['student_confidence'].std(), 4),
        'Mean Conf (Correct)': round(df.loc[df['correct'], 'student_confidence'].mean(), 4),
        'Mean Conf (Incorrect)': round(df.loc[~df['correct'], 'student_confidence'].mean(), 4),
        'ECE': round(float(ece_scores[model]), 4),
        'Fallback Rate': round(df['is_fallback'].mean(), 4),
        'Format Compliance': round(df['format_compliant'].mean(), 4),
    }
    for model, df in dfs.items()
}).T

summary_table